# Preview: memory-safe thumbnails of large rasters

When a raster is backed by dask (e.g. loaded lazily from Zarr or a stack of GeoTIFFs),
calling `.compute()` to visualize it can blow up your memory.  `xrspatial.preview()`
downsamples the data to a target pixel size using block averaging, and the whole
operation stays lazy until you ask for the result.  Peak memory is bounded by
the largest chunk plus the small output array.

This notebook generates a 1 TB dask-backed terrain raster and previews it at
1000x1000 pixels.  A `dask.distributed` LocalCluster is started so you can
watch the task graph and worker memory in the dashboard.

In [ ]:
import numpy as np
import xarray as xr
import dask.array as da
import matplotlib.pyplot as plt

import xrspatial
from xrspatial import generate_terrain, preview

In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit="2GB")
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")
client

## Generate a terrain tile

First, create a 1024x1024 terrain tile using `generate_terrain`.  This is the
building block we'll replicate into a massive dask array.

In [ ]:
# 1024x1024 in-memory terrain tile
canvas = xr.DataArray(np.zeros((1024, 1024), dtype=np.float32), dims=["y", "x"])
tile = generate_terrain(canvas, seed=12345)

fig, ax = plt.subplots(figsize=(6, 6))
tile.plot(ax=ax, cmap="terrain")
ax.set_title(f"Terrain tile ({tile.shape[0]}x{tile.shape[1]}, {tile.nbytes / 1e6:.1f} MB)")
ax.set_aspect("equal")
plt.tight_layout()

## Tile it into a 1 TB dask array

We replicate the tile 512x512 times using `dask.array.tile` to get a
524,288 x 524,288 raster.  At float32 that's 1.1 TB of data.  Nothing is
actually computed here -- dask just records the tiling as a lazy graph.

In [ ]:
# Tile the small terrain into a ~1 TB dask array
reps = 512
big_dask = da.tile(
    da.from_array(tile.values, chunks=(1024, 1024)),
    (reps, reps),
)
rows, cols = big_dask.shape
big = xr.DataArray(
    big_dask,
    dims=["y", "x"],
    coords={"y": np.arange(rows, dtype=np.float64), "x": np.arange(cols, dtype=np.float64)},
)

print(f"Shape:      {big.shape[0]:,} x {big.shape[1]:,}")
print(f"Chunk size: {big_dask.chunksize}")
print(f"Num chunks: {big_dask.numblocks}")
print(f"Total size: {big_dask.nbytes / 1e12:.2f} TB")
print(f"Dtype:      {big_dask.dtype}")

## Preview at 1000x1000

`preview()` builds a lazy coarsen-then-mean graph.  Calling `.compute()` on the
result materializes only the 1000x1000 output -- about 4 MB.

In [ ]:
%%time
small = preview(big, width=1000).compute()

print(f"Output shape: {small.shape}")
print(f"Output size:  {small.nbytes / 1e6:.1f} MB")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
small.plot(ax=ax, cmap="terrain")
ax.set_title(f"1000x1000 preview of a {big_dask.nbytes / 1e12:.1f} TB raster")
ax.set_aspect("equal")
plt.tight_layout()

## Different preview sizes

You can control both width and height.  Omitting height preserves the aspect ratio.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, w in zip(axes, [100, 500, 2000]):
    p = preview(big, width=w).compute()
    p.plot(ax=ax, cmap="terrain", add_colorbar=False)
    ax.set_title(f"{p.shape[0]}x{p.shape[1]} ({p.nbytes / 1e6:.1f} MB)")
    ax.set_aspect("equal")
plt.tight_layout()

## Accessor syntax

You can also call `preview` directly on a DataArray or Dataset via the `.xrs` accessor.

In [ ]:
# Accessor on a DataArray
small = big.xrs.preview(width=500).compute()
print(f"DataArray accessor: {small.shape}")

# Accessor on a Dataset
ds = xr.Dataset({"elevation": big, "slope_proxy": big * 0.1})
small_ds = ds.xrs.preview(width=500)
for name, var in small_ds.data_vars.items():
    print(f"Dataset var '{name}': {var.shape}")

In [ ]:
client.close()
cluster.close()